# HoTHP vs RoTHP: Three Targeted Scientific Tests

**Run on Colab: Runtime → Change runtime type → T4 GPU**

## Hypotheses

- **H1 — Scale Invariance**: HoTHP's `_normalize_timestamps()` makes the encoder *architecturally* scale-invariant.
  RoTHP has no such protection — fixed frequencies get the wrong phase when test timescale ≠ training timescale.
  We test this with two sub-experiments: (a) full-model NLL and (b) encoder cosine similarity, which isolates
  the attention mechanism from the intensity integral term in the loss.
- **H2 — Sample Efficiency**: HoTHP's monotonic-decay inductive bias should require less data to learn recency.
  We track both final NLL and epochs-to-convergence.
- **H3 — Long-Range Process**: When the true Hawkes kernel decays slowly (β_norm≈0.025), HoTHP's monotonic
  basis is a structural match. We quantify alignment between learned attention and the true kernel.

### Design notes
- Both models are trained with **2 random restarts** (best val kept) to reduce initialization sensitivity.
- H1 NLL test uses **global normalization** to expose scale shifts. This means `time_delta_seqs` in the
  loss are also scaled, which affects both models' direct intensity terms. The encoder cosine test
  controls for this by comparing representations directly.
- H2/H3 use **per-sequence normalization** (mean gap = 1.0) which is the standard fair baseline.

In [ ]:
import os
if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git
!pip install omegaconf -q

In [ ]:
import os, sys, math, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import contextlib

BASE_SEED = 42
N_SEEDS   = 5
N_RESTARTS = 2   # restarts per model per seed to reduce init sensitivity

def set_global_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def make_run_seed(*parts, base_seed=BASE_SEED):
    key = '::'.join(map(str, parts))
    return (base_seed + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_global_seed(BASE_SEED)
sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sys.path.insert(0, os.path.abspath('ufc-easytpp'))

import easy_tpp.model.torch_model.torch_baselayer as baselayer

def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        if mask.dim() == 3: mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None: p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

baselayer.attention = attention_fixed
import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = attention_fixed

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

NUM_TYPES = 2; pad_id = NUM_TYPES
config = ModelConfig(**{
    'hidden_size': 32, 'num_layers': 2, 'num_heads': 2, 'dropout_rate': 0.1,
    'num_event_types': NUM_TYPES, 'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': pad_id, 'time_emb_size': 32, 'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1, 'model_id': 'ScientificCase',
    'thinning': {'num_sample':1,'num_exp':500,'over_sample_rate':5.0,
                 'patience_counter':5,'num_samples_boundary':5,'dtime_max':5.0,'num_step_gen':1},
    'loss_integral_num_sample_per_step': 20, 'use_mc_samples': False,
})

USE_AMP = device.type == 'cuda'
if USE_AMP:
    try:
        _scaler_cls = torch.amp.GradScaler
        _autocast_fn = lambda: torch.amp.autocast(device_type='cuda')
    except AttributeError:
        _scaler_cls = torch.cuda.amp.GradScaler
        _autocast_fn = lambda: torch.cuda.amp.autocast()
else:
    _scaler_cls = None
    _autocast_fn = contextlib.nullcontext

print(f'Device: {device}  |  AMP: {USE_AMP}  |  Restarts per model: {N_RESTARTS}')

In [ ]:
# ── Process definitions ──────────────────────────────────────────────────────
#
# FAST-DECAY (baseline): beta_normalized ≈ 0.37
#   Influence at normalized lag 20: ~0.001  (negligible — both models should learn to ignore history)
#
# SLOW-DECAY (long-range): beta_normalized ≈ 0.025
#   Influence at normalized lag 20: ~0.61  (strong — model must encode long-range deps)
#   Influence at normalized lag 100: ~0.08  (still detectable at very long range)

PROC_FAST = dict(
    mu=np.array([0.4, 0.4]),
    alpha=np.array([[0.12, 0.08], [0.08, 0.12]]),
    beta=0.5,
    label='fast-decay (β_norm≈0.37)',
)
PROC_SLOW = dict(
    mu=np.array([0.3, 0.3]),
    alpha=np.array([[0.008, 0.006], [0.006, 0.008]]),
    beta=0.02,
    label='slow-decay (β_norm≈0.025)',
)

def generate_hawkes(rng, proc, horizon, min_events=20, max_events=150):
    mu, alpha, beta = proc['mu'], proc['alpha'], proc['beta']
    n_types = len(mu)
    for _ in range(50):
        events, t = [], 0.0
        while t < horizon and len(events) < max_events:
            intensity = mu.copy()
            for t_i, k_i in events:
                intensity += alpha[:, k_i] * np.exp(-beta * (t - t_i))
            lam_bar = float(np.sum(intensity))
            if lam_bar <= 1e-9: break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon: break
            candidate = mu.copy()
            for t_i, k_i in events:
                candidate += alpha[:, k_i] * np.exp(-beta * (t - t_i))
            lam_sum = float(np.sum(candidate))
            if rng.uniform() <= lam_sum / lam_bar:
                probs = candidate / lam_sum
                events.append((t, int(rng.choice(n_types, p=probs))))
        if len(events) >= min_events:
            return events[:max_events]
    return events[:max_events]

def make_dataset(rng, proc, n_seqs, horizon, min_events=20, max_events=150):
    return [generate_hawkes(rng, proc, horizon, min_events, max_events)
            for _ in range(n_seqs)]

# Verify beta_normalized for each process
rng0 = np.random.default_rng(BASE_SEED)
for proc in [PROC_FAST, PROC_SLOW]:
    seqs = make_dataset(rng0, proc, 200, horizon=50.0)
    all_gaps = []
    for seq in seqs:
        times = sorted([t for t, _ in seq])
        all_gaps.extend([times[i]-times[i-1] for i in range(1, len(times))])
    mean_gap = np.mean(all_gaps)
    beta_norm = proc['beta'] * mean_gap
    print(f"{proc['label']}")
    print(f"  mean_gap={mean_gap:.3f}, beta_norm={beta_norm:.4f}, "
          f"inf@20={math.exp(-beta_norm*20):.3f}, inf@100={math.exp(-beta_norm*100):.3f}")

In [ ]:
# ── Data utilities ───────────────────────────────────────────────────────────

def to_tensors_global_norm(seqs, global_mean_gap):
    """Normalise with a fixed global constant. Used for H1 to expose scale shifts."""
    converted = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        times  = torch.tensor([t for t, _ in seq], dtype=torch.float32)
        types  = torch.tensor([k for _, k in seq], dtype=torch.long)
        deltas = torch.zeros_like(times)
        deltas[1:] = times[1:] - times[:-1]
        times  = (times - times[0]) / global_mean_gap
        deltas = deltas / global_mean_gap
        converted.append({'time_seqs': times, 'time_delta_seqs': deltas, 'type_seqs': types})
    return converted

def to_tensors_per_seq(seqs):
    """Per-sequence normalisation (mean gap = 1.0). Used for H2/H3."""
    converted = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        times  = torch.tensor([t for t, _ in seq], dtype=torch.float32)
        types  = torch.tensor([k for _, k in seq], dtype=torch.long)
        deltas = torch.zeros_like(times)
        deltas[1:] = times[1:] - times[:-1]
        mean_gap = deltas[1:].mean().clamp(min=1e-6)
        times  = (times - times[0]) / mean_gap
        deltas = deltas / mean_gap
        converted.append({'time_seqs': times, 'time_delta_seqs': deltas, 'type_seqs': types})
    return converted

def collate_fn(batch_list):
    B = len(batch_list); L = max(len(x['time_seqs']) for x in batch_list)
    pad_time  = torch.zeros(B, L, dtype=torch.float32)
    pad_delta = torch.zeros(B, L, dtype=torch.float32)
    pad_type  = torch.full((B, L), pad_id, dtype=torch.long)
    npm       = torch.zeros(B, L, dtype=torch.float32)
    attn      = torch.ones(B, L, L, dtype=torch.bool)
    causal    = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
    for i, item in enumerate(batch_list):
        l = len(item['time_seqs'])
        pad_time[i, :l] = item['time_seqs']; pad_delta[i, :l] = item['time_delta_seqs']
        pad_type[i, :l] = item['type_seqs']; npm[i, :l] = 1.0
        m = causal.clone(); m[:, l:] = True; m[l:, :] = True; attn[i] = m
    return (pad_time, pad_delta, pad_type, npm, attn)

def make_loader(data, batch_size, shuffle=False, seed=None):
    gen = None
    if shuffle and seed is not None:
        gen = torch.Generator(); gen.manual_seed(seed)
    return DataLoader(data, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate_fn, generator=gen)

# ── Training utilities ───────────────────────────────────────────────────────

def evaluate_nll(model, loader):
    model.eval(); total_loss = total_events = 0
    with torch.no_grad():
        for batch in loader:
            batch = [t.to(device) for t in batch]
            with _autocast_fn():
                loss, num = model.loglike_loss(batch)
            total_loss += loss.item(); total_events += num
    return total_loss / (total_events + 1e-9)

def train_model(model, train_loader, val_loader, epochs=400, patience=20,
                grad_clip=1.0, lr=1e-3):
    """Train and return (best_val_nll, epochs_to_best)."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=8, min_lr=1e-5)
    scaler = _scaler_cls(enabled=True) if USE_AMP else None
    best_val, best_state, no_improve, best_ep = float('inf'), None, 0, 0
    for ep in range(epochs):
        model.train()
        for batch in train_loader:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            with _autocast_fn():
                loss, num = model.loglike_loss(batch)
                nll = loss / (num + 1e-9)
            if not torch.isnan(nll):
                if scaler:
                    scaler.scale(nll).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(opt); scaler.update()
                else:
                    nll.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    opt.step()
        val_nll = evaluate_nll(model, val_loader)
        sched.step(val_nll)
        if val_nll < best_val - 1e-4:
            best_val = val_nll
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0; best_ep = ep + 1
        else:
            no_improve += 1
        if no_improve >= patience: break
    if best_state: model.load_state_dict(best_state)
    return best_val, best_ep

def train_best_of(model_cls, n_restarts, train_loader, val_loader,
                  epochs=400, patience=20, lr=1e-3, base_seed=0):
    """
    Train n_restarts times with different init seeds, return the model
    with the best validation NLL. Reduces sensitivity to bad local minima.
    Returns (model, best_val_nll, epochs_to_best).
    """
    best_val, best_state, best_ep = float('inf'), None, 0
    for r in range(n_restarts):
        set_global_seed(base_seed + r * 7919)
        model = model_cls(config).to(device)
        val_nll, ep = train_model(model, train_loader, val_loader, epochs, patience, lr)
        if val_nll < best_val:
            best_val = val_nll
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_ep = ep
    model = model_cls(config).to(device)
    model.load_state_dict(best_state)
    return model, best_val, best_ep

# ── Analysis utilities ───────────────────────────────────────────────────────

def encoder_cosine_similarity(model, ref_data, scaled_data, n_seqs=64):
    """
    Compute mean cosine similarity between encoder outputs for the same
    sequences at scale=1 (ref_data) and at a different scale (scaled_data).

    For HoTHP: _normalize_timestamps cancels the scale -> cosine should be ~1.0.
    For RoTHP: rotary angles change with scale -> cosine drops below 1.0.

    This isolates the attention mechanism from the loss's direct intensity term.
    """
    model.eval()
    ref_loader = make_loader(ref_data[:n_seqs], 32)
    sc_loader  = make_loader(scaled_data[:n_seqs], 32)
    cos_sims = []
    with torch.no_grad():
        for ref_batch, sc_batch in zip(ref_loader, sc_loader):
            ref_batch = [t.to(device) for t in ref_batch]
            sc_batch  = [t.to(device) for t in sc_batch]
            t_ref, _, type_ref, mask_ref, attn_ref = ref_batch
            t_sc,  _, type_sc,  mask_sc,  attn_sc  = sc_batch
            enc_ref = model.forward(t_ref, type_ref, attn_ref)
            enc_sc  = model.forward(t_sc,  type_sc,  attn_sc)
            for b in range(enc_ref.shape[0]):
                sl = int(mask_ref[b].sum().item())
                r = enc_ref[b, :sl].reshape(-1)
                s = enc_sc[b,  :sl].reshape(-1)
                cos_sims.append(F.cosine_similarity(r.unsqueeze(0), s.unsqueeze(0)).item())
    return float(np.mean(cos_sims))

def bin_attn(lags, weights, n_bins=50):
    q98 = np.quantile(lags, 0.98)
    edges = np.linspace(0, q98, n_bins+1); centers = (edges[:-1]+edges[1:])/2
    means, stds = [], []
    for i in range(n_bins):
        m = (lags>=edges[i])&(lags<edges[i+1])
        means.append(weights[m].mean() if m.sum()>5 else np.nan)
        stds.append(weights[m].std() if m.sum()>5 else np.nan)
    return centers, np.array(means), np.array(stds)

def kernel_alignment(attn_mean, true_kernel):
    """
    Cosine similarity between the learned attention profile and the true Hawkes kernel.
    Both vectors are masked to remove NaN bins. Higher = better structural alignment.
    """
    valid = ~np.isnan(attn_mean)
    a = attn_mean[valid]; k = true_kernel[valid]
    if len(a) < 5: return float('nan')
    return float(np.dot(a, k) / (np.linalg.norm(a) * np.linalg.norm(k) + 1e-12))

print('Utilities ready.')

---
## H1 — Scale Invariance

**Setup**: Train with global normalization (training mean gap = 1.0).  
At test time, multiply raw timestamps by `s`, then apply the **same** global constant.
This means the model receives inter-event gaps of size `s` at test time.

**Expected for the attention mechanism:**
- **HoTHP**: `_normalize_timestamps()` divides by per-sequence mean gap = `s` → always sees gaps of size 1.0 → encoder output is identical → encoder cosine = 1.0 at all scales.
- **RoTHP**: fixed frequencies `θⱼ` → rotary phase `θⱼ × s × gap` → wrong phase at `s ≠ 1` → cosine drops.

**Important caveat for the NLL test**: `time_delta_seqs` (inter-event intervals) also flows into the
intensity integral in the loss function. Since we use global normalization, this term is also scaled
and is **not protected by `_normalize_timestamps`** in either model. This means both models will
degrade in NLL at extreme scales regardless. The encoder cosine test below controls for this.

In [ ]:
TEST_SCALES = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0]
seeds = [BASE_SEED + i * 100 for i in range(N_SEEDS)]

h1_results = []   # {seed, scale, rothp_nll, hothp_nll, rothp_val, hothp_val}
h1_models  = {}   # {seed: (rothp, hothp)} — kept for cosine test

rng_h1 = np.random.default_rng(make_run_seed('H1', 'data'))
train_raw = make_dataset(rng_h1, PROC_FAST, 500, horizon=20.0)
val_raw   = make_dataset(rng_h1, PROC_FAST, 100, horizon=20.0)
test_raw  = make_dataset(rng_h1, PROC_FAST, 200, horizon=20.0)

all_gaps = []
for seq in train_raw:
    times = sorted([t for t, _ in seq])
    all_gaps.extend([times[i]-times[i-1] for i in range(1, len(times))])
global_mean_gap = float(np.mean(all_gaps))
print(f'Global mean gap (training): {global_mean_gap:.4f}')

train_data = to_tensors_global_norm(train_raw, global_mean_gap)
val_data   = to_tensors_global_norm(val_raw,   global_mean_gap)
test_ref   = to_tensors_global_norm(test_raw,  global_mean_gap)  # scale=1 reference

for seed_idx, seed in enumerate(seeds):
    print(f'\n--- Seed {seed_idx+1}/{N_SEEDS} (seed={seed}) ---')
    train_loader = make_loader(train_data, 64, shuffle=True, seed=seed)
    val_loader   = make_loader(val_data,   64)

    # Train both with N_RESTARTS to reduce init sensitivity
    rothp, bv_r, ep_r = train_best_of(RoTHP, N_RESTARTS, train_loader, val_loader,
                                       epochs=400, patience=20, lr=1e-3,
                                       base_seed=make_run_seed('H1','RoTHP', base_seed=seed))
    hothp, bv_h, ep_h = train_best_of(HoTHP, N_RESTARTS, train_loader, val_loader,
                                       epochs=400, patience=20, lr=5e-4,
                                       base_seed=make_run_seed('H1','HoTHP', base_seed=seed))

    print(f'  Best val — RoTHP: {bv_r:.4f} (ep {ep_r})  HoTHP: {bv_h:.4f} (ep {ep_h})')
    h1_models[seed] = (rothp, hothp)

    for s in TEST_SCALES:
        test_scaled = [[(t * s, k) for t, k in seq] for seq in test_raw]
        test_data_s = to_tensors_global_norm(test_scaled, global_mean_gap)
        test_loader = make_loader(test_data_s, 64)
        r_nll = evaluate_nll(rothp, test_loader)
        h_nll = evaluate_nll(hothp, test_loader)
        h1_results.append({'seed': seed, 'scale': s,
                           'rothp_nll': r_nll, 'hothp_nll': h_nll,
                           'rothp_val': bv_r,  'hothp_val': bv_h})
        print(f'  scale={s:5.1f}x: RoTHP={r_nll:.4f}  HoTHP={h_nll:.4f}')

print('\nH1 NLL experiment complete.')

### H1b — Encoder Cosine Similarity

For each trained model, compare the **encoder output** (the hidden representation after the transformer stack)
for the same sequences at scale=1 vs each test scale.

- **HoTHP**: `_normalize_timestamps` compensates internally → the encoder sees the same normalised times
  regardless of input scale → cosine similarity should be ≈ 1.0 at all scales.
- **RoTHP**: the rotary embedding uses `time_seqs` directly → different phase at different scales
  → cosine similarity drops with scale distance from 1.

This is the **purest test** of architectural scale invariance, unconfounded by the loss function.

In [ ]:
h1_cosine_results = []  # {seed, scale, rothp_cos, hothp_cos}

for seed_idx, seed in enumerate(seeds):
    print(f'Seed {seed_idx+1}/{N_SEEDS}', end='  ')
    rothp, hothp = h1_models[seed]
    for s in TEST_SCALES:
        test_scaled = [[(t * s, k) for t, k in seq] for seq in test_raw]
        test_data_s = to_tensors_global_norm(test_scaled, global_mean_gap)
        r_cos = encoder_cosine_similarity(rothp, test_ref, test_data_s)
        h_cos = encoder_cosine_similarity(hothp, test_ref, test_data_s)
        h1_cosine_results.append({'seed': seed, 'scale': s,
                                   'rothp_cos': r_cos, 'hothp_cos': h_cos})
    print()

cos_df = pd.DataFrame(h1_cosine_results)
print('\nEncoder cosine similarity (mean across seeds):')
print(cos_df.groupby('scale')[['rothp_cos','hothp_cos']].mean().round(4).to_string())

In [ ]:
h1_df = pd.DataFrame(h1_results)
ref = h1_df[h1_df['scale']==1.0].set_index('seed')[['rothp_nll','hothp_nll']]
rows = []
for _, row in h1_df.iterrows():
    rows.append({'seed': row['seed'], 'scale': row['scale'],
                 'rothp_delta': row['rothp_nll'] - ref.loc[row['seed'],'rothp_nll'],
                 'hothp_delta': row['hothp_nll'] - ref.loc[row['seed'],'hothp_nll']})
delta_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Absolute NLL
for col, label, color in [('rothp_nll','RoTHP','#4C72B0'),('hothp_nll','HoTHP','#C44E52')]:
    grp = h1_df.groupby('scale')[col]
    m, s = grp.mean(), grp.std()
    axes[0].plot(np.log10(TEST_SCALES), m.values, 'o-', color=color, lw=2, ms=7, label=label)
    axes[0].fill_between(np.log10(TEST_SCALES), m-s, m+s, color=color, alpha=0.12)
axes[0].axvline(0, color='gray', ls='--', alpha=0.6, label='training scale (1×)')
axes[0].set_xticks(np.log10(TEST_SCALES))
axes[0].set_xticklabels([f'{s}×' for s in TEST_SCALES])
axes[0].set_xlabel('Test scale'); axes[0].set_ylabel('Test NLL')
axes[0].set_title('H1a: NLL vs scale\n(confounded by intensity loss term)')
axes[0].legend()

# Panel 2: ΔNLL
for col, label, color in [('rothp_delta','RoTHP','#4C72B0'),('hothp_delta','HoTHP','#C44E52')]:
    grp = delta_df.groupby('scale')[col]
    m, s = grp.mean(), grp.std()
    axes[1].plot(np.log10(TEST_SCALES), m.values, 'o-', color=color, lw=2, ms=7, label=label)
    axes[1].fill_between(np.log10(TEST_SCALES), m-s, m+s, color=color, alpha=0.12)
axes[1].axhline(0, color='gray', ls='--', alpha=0.6)
axes[1].axvline(0, color='gray', ls='--', alpha=0.4)
axes[1].set_xticks(np.log10(TEST_SCALES))
axes[1].set_xticklabels([f'{s}×' for s in TEST_SCALES])
axes[1].set_xlabel('Test scale')
axes[1].set_ylabel('ΔNLL relative to scale=1×')
axes[1].set_title('H1a: NLL degradation from scale shift')
axes[1].legend()

# Panel 3: Encoder cosine similarity (the clean architectural test)
for col, label, color in [('rothp_cos','RoTHP','#4C72B0'),('hothp_cos','HoTHP','#C44E52')]:
    grp = cos_df.groupby('scale')[col]
    m, s = grp.mean(), grp.std()
    axes[2].plot(np.log10(TEST_SCALES), m.values, 'o-', color=color, lw=2, ms=7, label=label)
    axes[2].fill_between(np.log10(TEST_SCALES), m-s, m+s, color=color, alpha=0.12)
axes[2].axhline(1.0, color='gray', ls='--', alpha=0.6, label='perfect invariance')
axes[2].axvline(0, color='gray', ls='--', alpha=0.4)
axes[2].set_xticks(np.log10(TEST_SCALES))
axes[2].set_xticklabels([f'{s}×' for s in TEST_SCALES])
axes[2].set_xlabel('Test scale')
axes[2].set_ylabel('Cosine similarity with scale=1× encoder output')
axes[2].set_title('H1b: Encoder cosine similarity\n(architectural test — unconfounded)')
axes[2].set_ylim(0, 1.05)
axes[2].legend()

plt.suptitle('H1 — Scale Invariance: HoTHP encoder is protected by _normalize_timestamps;\n'
             'RoTHP sees wrong oscillation frequency at shifted scales', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('H1_Scale_Invariance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nH1 ΔNLL summary (mean across seeds):')
print(delta_df.groupby('scale')[['rothp_delta','hothp_delta']].mean().round(4).to_string())
print('\nH1 encoder cosine (mean across seeds):')
print(cos_df.groupby('scale')[['rothp_cos','hothp_cos']].mean().round(4).to_string())

---
## H2+H3 — Sample Efficiency on Fast vs Slow Process

**H2**: HoTHP's inductive bias (monotonic decay ≡ recent events matter more) should reduce the amount
of training data needed. We measure both final NLL and **epochs-to-best** as a convergence speed proxy.

**H3**: This effect should be **larger for the slow-decay process**, where the model must represent
long-range influence (β_norm≈0.025, influence at lag 20 ≈ 0.61). HoTHP's hyperbolic kernel is a direct
structural match. RoTHP must superimpose multiple cosines to approximate monotonic decay — a harder
optimization problem, especially at small N.

In [ ]:
N_TRAIN_SIZES = [25, 50, 100, 250, 500]

rng_h23 = np.random.default_rng(make_run_seed('H2H3', 'data'))
datasets_h23 = {}
for proc_name, proc, horizon in [('fast', PROC_FAST, 20.0), ('slow', PROC_SLOW, 80.0)]:
    train_all = make_dataset(rng_h23, proc, 500, horizon, min_events=20, max_events=150)
    val_raw   = make_dataset(rng_h23, proc, 150, horizon, min_events=20, max_events=150)
    test_raw  = make_dataset(rng_h23, proc, 200, horizon, min_events=20, max_events=150)
    datasets_h23[proc_name] = {
        'train_all': to_tensors_per_seq(train_all),
        'val':       to_tensors_per_seq(val_raw),
        'test':      to_tensors_per_seq(test_raw),
        'label':     proc['label'],
    }
    lens = [len(s) for s in train_all]
    print(f'{proc_name}: len mean={np.mean(lens):.0f}, min={min(lens)}, max={max(lens)}')

h23_results = []  # {proc, n_train, seed, rothp_nll, hothp_nll, advantage, rothp_ep, hothp_ep}

for proc_name in ['fast', 'slow']:
    d = datasets_h23[proc_name]
    val_loader  = make_loader(d['val'],  64)
    test_loader = make_loader(d['test'], 64)
    print(f'\n=== {d["label"]} ===')
    for n_train in N_TRAIN_SIZES:
        train_subset = d['train_all'][:n_train]
        for seed in seeds:
            loader_seed = make_run_seed('H23', proc_name, n_train, seed)
            train_loader = make_loader(train_subset, min(32, n_train),
                                       shuffle=True, seed=loader_seed)

            rothp, r_val, r_ep = train_best_of(
                RoTHP, N_RESTARTS, train_loader, val_loader, epochs=400, patience=25, lr=1e-3,
                base_seed=make_run_seed('H23', proc_name, 'RoTHP', n_train, base_seed=seed))
            r_nll = evaluate_nll(rothp, test_loader)

            hothp, h_val, h_ep = train_best_of(
                HoTHP, N_RESTARTS, train_loader, val_loader, epochs=400, patience=25, lr=5e-4,
                base_seed=make_run_seed('H23', proc_name, 'HoTHP', n_train, base_seed=seed))
            h_nll = evaluate_nll(hothp, test_loader)

            h23_results.append({
                'proc': proc_name, 'n_train': n_train, 'seed': seed,
                'rothp_nll': r_nll, 'hothp_nll': h_nll,
                'advantage': r_nll - h_nll,
                'rothp_ep': r_ep,   'hothp_ep': h_ep,
            })

        grp = [r for r in h23_results if r['proc']==proc_name and r['n_train']==n_train]
        r_m = np.mean([r['rothp_nll'] for r in grp])
        h_m = np.mean([r['hothp_nll'] for r in grp])
        r_ep_m = np.mean([r['rothp_ep'] for r in grp])
        h_ep_m = np.mean([r['hothp_ep'] for r in grp])
        print(f'  N={n_train:4d}: RoTHP={r_m:.4f}(ep{r_ep_m:.0f})  '
              f'HoTHP={h_m:.4f}(ep{h_ep_m:.0f})  adv={r_m-h_m:+.4f}')

print('\nH2+H3 complete.')

In [ ]:
from scipy import stats

h23_df = pd.DataFrame(h23_results)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row_idx, proc_name in enumerate(['fast', 'slow']):
    proc_df = h23_df[h23_df['proc']==proc_name]
    label = datasets_h23[proc_name]['label']

    # Panel 1: NLL vs N_train
    ax = axes[row_idx, 0]
    for col, model_label, color in [('rothp_nll','RoTHP','#4C72B0'),('hothp_nll','HoTHP','#C44E52')]:
        grp = proc_df.groupby('n_train')[col]
        m, s = grp.mean(), grp.std()
        ax.plot(m.index, m.values, 'o-', color=color, lw=2, ms=7, label=model_label)
        ax.fill_between(m.index, m-s, m+s, color=color, alpha=0.12)
    ax.set_xscale('log'); ax.set_xlabel('N training (log)'); ax.set_ylabel('Test NLL')
    ax.set_title(f'NLL vs N_train\n{label}'); ax.legend(); ax.grid(True, alpha=0.3)

    # Panel 2: HoTHP advantage
    ax = axes[row_idx, 1]
    grp = proc_df.groupby('n_train')['advantage']
    m, s = grp.mean(), grp.std()
    colors_bar = ['#55A868' if v > 0 else '#C44E52' for v in m.values]
    ax.bar(range(len(m)), m.values, yerr=s.values, color=colors_bar, capsize=4, alpha=0.8)
    ax.axhline(0, color='gray', ls='--', alpha=0.7)
    ax.set_xticks(range(len(m)))
    ax.set_xticklabels([f'N={n}' for n in m.index])
    ax.set_ylabel('HoTHP advantage (RoTHP−HoTHP NLL)')
    ax.set_title(f'HoTHP advantage\n{label}')
    ax.grid(True, alpha=0.3, axis='y')

    for n in N_TRAIN_SIZES:
        sub = proc_df[proc_df['n_train']==n]
        if len(sub) >= 3:
            t, p2 = stats.ttest_rel(sub['rothp_nll'].values, sub['hothp_nll'].values)
            p1 = p2/2 if t > 0 else 1 - p2/2
            print(f'{proc_name} N={n}: adv={sub["advantage"].mean():+.4f}, p(one-sided)={p1:.3f}')

    # Panel 3: Epochs-to-best (convergence speed proxy)
    ax = axes[row_idx, 2]
    for col, model_label, color in [('rothp_ep','RoTHP','#4C72B0'),('hothp_ep','HoTHP','#C44E52')]:
        grp = proc_df.groupby('n_train')[col]
        m, s = grp.mean(), grp.std()
        ax.plot(m.index, m.values, 'o-', color=color, lw=2, ms=7, label=model_label)
        ax.fill_between(m.index, m-s, m+s, color=color, alpha=0.12)
    ax.set_xscale('log'); ax.set_xlabel('N training (log)')
    ax.set_ylabel('Epoch of best val NLL')
    ax.set_title(f'Convergence speed (H2)\n{label}\n(lower = faster)')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('H2+H3: Sample Efficiency and Convergence Speed — fast vs slow process',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('H2H3_Sample_Efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

---
## H3 Supporting Evidence: Attention Profile vs True Hawkes Kernel

We train one pair of models on full N=500 for each process and compare the learned attention profile
against the true Hawkes kernel. We report **kernel alignment** (cosine similarity between the learned
attention and the true kernel) as a quantitative metric.

**Expected**: HoTHP's monotonic attention profile should have higher kernel alignment,
especially on the slow-decay process where the true kernel persists over many lags.

In [ ]:
def extract_attention_vs_lag(model, loader, model_type, max_batches=30):
    model.eval()
    lags, weights = [], []
    with torch.no_grad():
        for b_idx, batch in enumerate(loader):
            if b_idx >= max_batches: break
            batch = [t.to(device) for t in batch]
            pad_time, pad_delta, pad_type, mask, attn_mask = batch
            enc = model.layer_type_emb(pad_type)
            layer = model.stack_layers[0]
            if model_type == 'rothp':
                cos, sin = model.rotary_emb(pad_time)
                _, attn_w = layer.self_attn(enc, enc, enc, attn_mask,
                                            cos=cos, sin=sin, output_weight=True)
            else:
                nt = model._normalize_timestamps(pad_time)
                _, attn_w = layer.self_attn(enc, enc, enc, attn_mask,
                                            time_seqs=nt,
                                            thetas=model.hope_emb.thetas,
                                            theta_prime=model.hope_emb.theta_prime,
                                            output_weight=True)
            attn_w = attn_w.mean(dim=1).cpu()
            pt = pad_time.cpu(); m = mask.cpu()
            for b in range(pt.shape[0]):
                sl = int(m[b].sum().item())
                for i in range(sl):
                    for j in range(i):
                        lag = float(pt[b, i] - pt[b, j])
                        if lag > 0:
                            lags.append(lag); weights.append(float(attn_w[b, i, j]))
    return np.array(lags), np.array(weights)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
alignment_results = {}

for row_idx, (proc_name, proc_def) in enumerate([('fast',PROC_FAST),('slow',PROC_SLOW)]):
    d = datasets_h23[proc_name]
    test_loader  = make_loader(d['test'], 32)
    val_loader   = make_loader(d['val'],  64)
    train_loader = make_loader(d['train_all'], 64, shuffle=True,
                               seed=make_run_seed('profile', proc_name))

    rothp_p, bv_r, _ = train_best_of(
        RoTHP, N_RESTARTS, train_loader, val_loader, epochs=400, patience=25, lr=1e-3,
        base_seed=make_run_seed('profile', proc_name, 'RoTHP'))
    hothp_p, bv_h, _ = train_best_of(
        HoTHP, N_RESTARTS, train_loader, val_loader, epochs=400, patience=25, lr=5e-4,
        base_seed=make_run_seed('profile', proc_name, 'HoTHP'))

    print(f'{proc_name}: RoTHP val={bv_r:.4f}  HoTHP val={bv_h:.4f}')

    r_lags, r_w = extract_attention_vs_lag(rothp_p, test_loader, 'rothp')
    h_lags, h_w = extract_attention_vs_lag(hothp_p, test_loader, 'hothp')
    rc, rm, rs = bin_attn(r_lags, r_w)
    hc, hm, hs = bin_attn(h_lags, h_w)

    # True Hawkes kernel (estimated beta_norm from test data)
    mean_gap_test = float(np.mean([float(item['time_delta_seqs'][1:].mean())
                                   for item in d['test'][:50]]))
    beta_norm = proc_def['beta'] * mean_gap_test
    true_kernel = np.exp(-beta_norm * rc); true_kernel /= true_kernel.max()

    # Kernel alignment metric
    r_align = kernel_alignment(rm, true_kernel)
    h_align = kernel_alignment(hm, true_kernel)
    alignment_results[proc_name] = {'rothp': r_align, 'hothp': h_align}
    print(f'  Kernel alignment — RoTHP: {r_align:.4f}  HoTHP: {h_align:.4f}')

    # Panel 1: Raw attention profiles
    ax = axes[row_idx, 0]
    ax.plot(rc, rm, color='#4C72B0', lw=1.8, label='RoTHP')
    ax.fill_between(rc, rm-rs, rm+rs, color='#4C72B0', alpha=0.12)
    ax.plot(hc, hm, color='#C44E52', lw=1.8, label='HoTHP')
    ax.fill_between(hc, hm-hs, hm+hs, color='#C44E52', alpha=0.12)
    ax.set_xlabel('Temporal lag (normalised)'); ax.set_ylabel('Mean attention weight')
    ax.set_title(f'Attention profile\n{proc_def["label"]}'); ax.legend(); ax.grid(True, alpha=0.3)

    # Panel 2: Normalised vs true kernel
    ax = axes[row_idx, 1]
    valid = ~np.isnan(hm)
    hm_n = hm[valid]/np.nanmax(np.abs(hm)) if np.nanmax(np.abs(hm))>0 else hm[valid]
    rm_n = rm[valid]/np.nanmax(np.abs(rm)) if np.nanmax(np.abs(rm))>0 else rm[valid]
    ax.plot(rc[valid], true_kernel[valid], 'k--', lw=2, label='True kernel')
    ax.plot(rc[valid], rm_n, color='#4C72B0', lw=1.8, label=f'RoTHP (align={r_align:.3f})')
    ax.plot(rc[valid], hm_n, color='#C44E52', lw=1.8, label=f'HoTHP (align={h_align:.3f})')
    ax.set_xlabel('Temporal lag (normalised)'); ax.set_ylabel('Normalised value')
    ax.set_title(f'Attention vs true kernel\n{proc_def["label"]}'); ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Panel 3: Alignment bar chart
    ax = axes[row_idx, 2]
    ax.bar(['RoTHP', 'HoTHP'], [r_align, h_align],
           color=['#4C72B0', '#C44E52'], alpha=0.8)
    ax.set_ylabel('Kernel alignment (cosine similarity with true kernel)')
    ax.set_title(f'Kernel alignment (H3)\n{proc_def["label"]}\n1.0 = perfect match')
    ax.set_ylim(0, 1.05)
    ax.axhline(1.0, color='gray', ls='--', alpha=0.4)
    for bar_i, val in enumerate([r_align, h_align]):
        ax.text(bar_i, val + 0.01, f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('H3: Attention profile vs true Hawkes kernel\n'
             'Kernel alignment = cosine similarity between learned attention and true exp(-β×lag)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('H3_Attention_vs_Kernel.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=' * 65)
print('SUMMARY: Three targeted tests for HoTHP vs RoTHP')
print('=' * 65)

# H1a — NLL
h1_pivot = delta_df.groupby('scale')[['rothp_delta','hothp_delta']].mean()
max_r_nll = h1_pivot['rothp_delta'].abs().max()
max_h_nll = h1_pivot['hothp_delta'].abs().max()
print(f'\nH1a — Scale Invariance (NLL):')
print(f'  Max RoTHP NLL degradation: {max_r_nll:+.4f}')
print(f'  Max HoTHP NLL degradation: {max_h_nll:+.4f}')
print(f'  Note: both degrade due to direct intensity term in loss (not protected by _normalize_timestamps)')

# H1b — Encoder cosine
cos_pivot = cos_df.groupby('scale')[['rothp_cos','hothp_cos']].mean()
min_r_cos = cos_pivot['rothp_cos'].min()
min_h_cos = cos_pivot['hothp_cos'].min()
print(f'\nH1b — Scale Invariance (encoder cosine):')
print(f'  Min RoTHP encoder cosine (worst scale): {min_r_cos:.4f}')
print(f'  Min HoTHP encoder cosine (worst scale): {min_h_cos:.4f}')
h1b_confirmed = min_h_cos > min_r_cos + 0.1
print(f'  → H1b {"CONFIRMED" if h1b_confirmed else "NOT CONFIRMED"}: '
      f'HoTHP encoder more stable across scales')

# H2+H3
print(f'\nH2+H3 — Sample Efficiency:')
for proc_name in ['fast', 'slow']:
    sub = h23_df[h23_df['proc']==proc_name]
    by_n = sub.groupby('n_train')['advantage'].mean()
    ep_r = sub.groupby('n_train')['rothp_ep'].mean()
    ep_h = sub.groupby('n_train')['hothp_ep'].mean()
    print(f'  {proc_name.upper()} process:')
    for n in N_TRAIN_SIZES:
        adv = by_n[n]; bar = '▓'*max(0,int(adv*200)) if adv>0 else '░'*max(0,int(-adv*200))
        print(f'    N={n:4d}: NLL adv={adv:+.4f} {bar}  '
              f'(RoTHP ep={ep_r[n]:.0f}, HoTHP ep={ep_h[n]:.0f})')

fast_adv = h23_df[h23_df['proc']=='fast'].groupby('n_train')['advantage'].mean()
slow_adv = h23_df[h23_df['proc']=='slow'].groupby('n_train')['advantage'].mean()
print(f'\n  slow mean advantage: {slow_adv.mean():+.4f}')
print(f'  fast mean advantage: {fast_adv.mean():+.4f}')
print(f'  → H3 {"SUPPORTED" if slow_adv.mean() > fast_adv.mean() else "NOT SUPPORTED"}: '
      f'advantage larger on slow-decay process')
print(f'  → H2 {"SUPPORTED" if slow_adv.iloc[0] > 0 else "NOT SUPPORTED"}: '
      f'HoTHP wins at small N on slow process')

# H3 kernel alignment
print(f'\nH3 — Kernel Alignment (cosine with true Hawkes kernel):')
for proc_name, res in alignment_results.items():
    winner = 'HoTHP' if res['hothp'] > res['rothp'] else 'RoTHP'
    print(f'  {proc_name.upper()}: RoTHP={res["rothp"]:.4f}  HoTHP={res["hothp"]:.4f}  → {winner} better aligned')